# Niveau 3 — Inspection des candidats

Classer chaque candidat en `plausible`, `faux_positif` ou `incertain`.


In [ ]:

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image


ROOT = Path.cwd()

CSV_PATH = (
    ROOT
    / "data"
    / "niveau3_pilote"
    / "candidats"
    / "candidats_uniques.csv"
)

CROPS_DIR = (
    ROOT
    / "data"
    / "niveau3_pilote"
    / "candidats"
    / "crops"
)

CONTEXT_DIR = (
    ROOT
    / "data"
    / "niveau3_pilote"
    / "candidats"
    / "contextes"
)


# ============================================================
# CHARGEMENT ROBUSTE
# ============================================================

df = pd.read_csv(CSV_PATH)

# Correctif définitif contre le bug Pandas float64
if "review_status" not in df.columns:
    df["review_status"] = ""

if "review_comment" not in df.columns:
    df["review_comment"] = ""

df["review_status"] = (
    df["review_status"]
    .fillna("")
    .astype("object")
)

df["review_comment"] = (
    df["review_comment"]
    .fillna("")
    .astype("object")
)


# ============================================================
# INDEX COURANT
# ============================================================

current = 0

# Aller directement au premier candidat non annoté
for i, value in enumerate(df["review_status"].astype(str)):
    if value.strip() == "":
        current = i
        break


# ============================================================
# WIDGETS
# ============================================================

output = widgets.Output()

comment = widgets.Textarea(
    placeholder="Commentaire optionnel...",
    description="Commentaire :",
    layout=widgets.Layout(width="700px", height="80px"),
)

btn_plausible = widgets.Button(
    description="✅ Plausible",
    button_style="success",
)

btn_false = widgets.Button(
    description="❌ Faux positif",
    button_style="danger",
)

btn_uncertain = widgets.Button(
    description="❓ Incertain",
    button_style="warning",
)

btn_prev = widgets.Button(
    description="⬅ Précédent",
)

btn_next = widgets.Button(
    description="Suivant ➡",
)


# ============================================================
# SAUVEGARDE
# ============================================================

def save():
    df.to_csv(
        CSV_PATH,
        index=False,
    )


# ============================================================
# AFFICHAGE
# ============================================================

def show_candidate():
    global current

    with output:
        clear_output(wait=True)

        row = df.iloc[current]

        candidate_id = str(row["candidate_id"])
        unique_id = str(row["unique_candidate_id"])

        context_path = CONTEXT_DIR / f"{candidate_id}.jpg"
        crop_path = CROPS_DIR / f"{candidate_id}.jpg"

        print(
            f"Candidat {current + 1}/{len(df)} "
            f"| {unique_id} "
            f"| détection {candidate_id}"
        )

        print(
            f"Confiance : {float(row['confidence']):.3f}"
        )

        print(
            f"Distance ANFR : "
            f"{float(row['nearest_ANFR_distance_m']):.1f} m"
        )

        print(
            f"GPS : "
            f"{float(row['latitude']):.6f}, "
            f"{float(row['longitude']):.6f}"
        )

        print(
            "Cluster :",
            row["cluster_candidate_ids"]
        )

        print(
            "Statut actuel :",
            row["review_status"]
            if str(row["review_status"]).strip()
            else "non annoté"
        )

        comment.value = (
            ""
            if pd.isna(row["review_comment"])
            else str(row["review_comment"])
        )

        fig, ax = plt.subplots(
            1,
            2,
            figsize=(12, 6),
        )

        if context_path.exists():
            context = Image.open(context_path)
            ax[0].imshow(context)
            ax[0].set_title("Contexte")
            ax[0].axis("off")

        if crop_path.exists():
            crop = Image.open(crop_path)
            ax[1].imshow(crop)
            ax[1].set_title("Crop détecté")
            ax[1].axis("off")

        plt.tight_layout()
        plt.show()


# ============================================================
# ACTIONS
# ============================================================

def set_status(status):
    global current

    df.at[current, "review_status"] = status
    df.at[current, "review_comment"] = comment.value

    save()

    if current < len(df) - 1:
        current += 1

    show_candidate()


def on_plausible(_):
    set_status("plausible")


def on_false(_):
    set_status("faux_positif")


def on_uncertain(_):
    set_status("incertain")


def on_prev(_):
    global current

    df.at[current, "review_comment"] = comment.value
    save()

    if current > 0:
        current -= 1

    show_candidate()


def on_next(_):
    global current

    df.at[current, "review_comment"] = comment.value
    save()

    if current < len(df) - 1:
        current += 1

    show_candidate()


btn_plausible.on_click(on_plausible)
btn_false.on_click(on_false)
btn_uncertain.on_click(on_uncertain)
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)


buttons = widgets.HBox(
    [
        btn_plausible,
        btn_false,
        btn_uncertain,
        btn_prev,
        btn_next,
    ]
)


print("Candidats :", len(df))
print(
    "Déjà examinés :",
    (df["review_status"].astype(str).str.strip() != "").sum()
)

display(output)
display(comment)
display(buttons)

show_candidate()
